In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv


In [2]:
df= pd.read_csv("/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv")

In [3]:
df.head()
df['State'].value_counts()

State
CA    1741433
FL     880192
TX     582837
SC     382557
NY     347960
NC     338199
VA     303301
PA     296620
MN     192084
OR     179660
AZ     170609
GA     169234
IL     168958
TN     167388
MI     162191
LA     149701
NJ     140719
MD     140417
OH     118115
WA     108221
AL     101044
UT      97079
CO      90885
OK      83647
MO      77323
CT      71005
IN      67224
MA      61996
WI      34688
KY      32254
NE      28870
MT      28496
IA      26307
AR      22780
NV      21665
KS      20992
DC      18630
RI      16971
MS      15181
DE      14097
WV      13793
ID      11376
NM      10325
NH      10213
WY       3757
ND       3487
ME       2698
VT        926
SD        289
Name: count, dtype: int64

In [4]:
df_tx= df[df['State'] == 'TX']

In [5]:
df_tx.shape

(582837, 46)

In [6]:
columns= ['Severity', 'Start_Time', 'Start_Lat','Start_Lng',  'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)','Precipitation(in)', 'Weather_Condition',  'Crossing',  'Junction', 'Railway',  'Stop', 'Traffic_Signal', 'Sunrise_Sunset']

In [7]:

df_tx=df_tx[columns]

In [8]:
df_tx.columns

Index(['Severity', 'Start_Time', 'Start_Lat', 'Start_Lng', 'Temperature(F)',
       'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)',
       'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Crossing',
       'Junction', 'Railway', 'Stop', 'Traffic_Signal', 'Sunrise_Sunset'],
      dtype='object')

In [9]:
df_tx.notnull().sum()

Severity             582837
Start_Time           582837
Start_Lat            582837
Start_Lng            582837
Temperature(F)       574396
Wind_Chill(F)        356841
Humidity(%)          573610
Pressure(in)         575107
Visibility(mi)       573645
Wind_Speed(mph)      541510
Precipitation(in)    353507
Weather_Condition    573638
Crossing             582837
Junction             582837
Railway              582837
Stop                 582837
Traffic_Signal       582837
Sunrise_Sunset       582394
dtype: int64

In [10]:
df_tx.drop(columns=['Wind_Chill(F)', 'Precipitation(in)'], inplace=True)

In [11]:
df_tx.fillna({
    'Temperature(F)': df_tx['Temperature(F)'].mean(),
    'Humidity(%)': df_tx['Humidity(%)'].mean(),
    'Pressure(in)': df_tx['Pressure(in)'].mean(),
    'Visibility(mi)': df_tx['Visibility(mi)'].mean(),
    'Wind_Speed(mph)': df_tx['Wind_Speed(mph)'].mean()
},inplace=True)

In [12]:
df_tx.fillna({
    'Weather_Condition': df_tx['Weather_Condition'].mode()[0],
    'Sunrise_Sunset': df_tx['Sunrise_Sunset'].mode()[0]
},inplace=True)

In [13]:
df_tx['Start_Time'] = pd.to_datetime(df_tx['Start_Time'], errors='coerce')
df_tx['hour']= df_tx['Start_Time'].dt.hour
df_tx['day']= df_tx['Start_Time'].dt.dayofweek
df_tx['month']= df_tx['Start_Time'].dt.month

In [14]:
df_tx.sample(12)
df_tx.notnull().sum()
df_tx.fillna({
    'hour': df_tx['hour'].mode()[0],
    'day': df_tx['day'].mode()[0],
    'month': df_tx['month'].mode()[0]
}, inplace=True)

In [15]:
df_tx.notnull().sum()
df_tx.drop(columns= ['Start_Time'], inplace=True)

In [16]:

df_tx.notnull().sum()

Severity             582837
Start_Lat            582837
Start_Lng            582837
Temperature(F)       582837
Humidity(%)          582837
Pressure(in)         582837
Visibility(mi)       582837
Wind_Speed(mph)      582837
Weather_Condition    582837
Crossing             582837
Junction             582837
Railway              582837
Stop                 582837
Traffic_Signal       582837
Sunrise_Sunset       582837
hour                 582837
day                  582837
month                582837
dtype: int64

In [17]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

ohe= OneHotEncoder(handle_unknown='ignore',sparse_output=False)

In [18]:
cat_cols= ['Weather_Condition',  'Crossing',  'Junction', 'Railway',  'Stop', 'Traffic_Signal', 'Sunrise_Sunset']
preprocessor= ColumnTransformer(transformers=[
    ('cat', ohe, cat_cols)
], remainder= 'passthrough')   
 

In [19]:
x= df_tx.drop(columns=['Severity'])
y= df_tx['Severity']


In [20]:
df_tx['Severity'].value_counts()

Severity
2    450952
3    120443
4      7209
1      4233
Name: count, dtype: int64

In [21]:
y1=df_tx['Severity']-1

In [22]:
y1.value_counts()

Severity
1    450952
2    120443
3      7209
0      4233
Name: count, dtype: int64

In [23]:
y1 = (df_tx['Severity'] >= 3).astype(int)

In [24]:
y1.value_counts()

Severity
0    455185
1    127652
Name: count, dtype: int64

In [29]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

x_train, x_test, y_train, y_test = train_test_split(
    x, y1, test_size=0.2, random_state=42
)


# pipeline
model2 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        objective='binary:logistic',
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=3,
        gamma=0.1,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=42,
        scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
    ))
])

# train
model2.fit(x_train, y_train)

# predict (0 or 1)
# y_pred = model2.predict(x_test)
y_prob = model2.predict_proba(x_test)[:, 1]
y_pred = (y_prob > 0.5).astype(int)

# evaluate
print("Accuracy:", model2.score(x_test, y_test))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8112775375746345

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.79      0.87     91005
           1       0.54      0.88      0.67     25563

    accuracy                           0.81    116568
   macro avg       0.75      0.83      0.77    116568
weighted avg       0.87      0.81      0.82    116568


Confusion Matrix:
 [[72165 18840]
 [ 3159 22404]]


In [30]:
import joblib
joblib.dump(model2,"tx_model.pkl")

['tx_model.pkl']

In [25]:
import numpy as np

df_tx = df_tx.copy()

# remove impossible coordinates
df_tx = df_tx[
    (df_tx['Start_Lat'].between(-90, 90)) &
    (df_tx['Start_Lng'].between(-180, 180))
]

# remove impossible weather values
df_tx = df_tx[df_tx['Temperature(F)'] > -100]   # sanity check
df_tx = df_tx[df_tx['Wind_Speed(mph)'] >= 0]
df_tx = df_tx[df_tx['Pressure(in)'] > 0]
df_tx = df_tx[df_tx['Humidity(%)'].between(0, 100)]
df_tx = df_tx[df_tx['Visibility(mi)'] >= 0]

In [26]:
def iqr_filter(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    return df[(df[col] >= lower) & (df[col] <= upper)]

numeric_cols = [
    'Temperature(F)',
    'Wind_Speed(mph)',
    'Pressure(in)',
    'Visibility(mi)',
    'Humidity(%)'
]

for col in numeric_cols:
    df_tx = iqr_filter(df_tx, col)

In [27]:
lat_min, lat_max = df_tx['Start_Lat'].quantile([0.01, 0.99])
lng_min, lng_max = df_tx['Start_Lng'].quantile([0.01, 0.99])

df_tx = df_tx[
    (df_tx['Start_Lat'].between(lat_min, lat_max)) &
    (df_tx['Start_Lng'].between(lng_min, lng_max))
]

In [28]:
df_tx.shape

(422586, 18)